In [1]:
import json
import os
import re
import subprocess
from pathlib import Path
from urllib.parse import urlparse

try:
    import pandas as pd
    PANDAS_AVAILABLE = True
except ImportError:
    PANDAS_AVAILABLE = False

In [2]:
class RecomputationAvoidanceChecker:
    """
    RecomputationAvoidanceChecker

    This checker evaluates whether a research software artifact supports
    mechanisms that reduce unnecessary repeated computation.

    It checks for evidence of:
    - caching
    - checkpointing
    - saved intermediate outputs
    - workflow files
    - parameter/config files
    - modular scripts
    - notebooks
    - resume/restart functionality

    Formal idea:
        recomputationAvoidance : A → {True, False}
    """

    def __init__(
        self,
        json_file,
        download_dir="downloads",
        minimum_score=4
    ):
        self.json_file = json_file
        self.download_dir = Path(download_dir)
        self.download_dir.mkdir(parents=True, exist_ok=True)

        self.minimum_score = minimum_score
        self.artifacts = self.load_metadata(json_file)
        self.results = []

    def load_metadata(self, json_file):
        with open(json_file, "r", encoding="utf-8") as file:
            data = json.load(file)

        return data.get("artifacts", {})

    def is_git_repository(self, uri):
        return isinstance(uri, str) and uri.startswith("https://github.com/")

    def repo_name_from_uri(self, uri):
        parsed = urlparse(uri)
        repo_name = parsed.path.rstrip("/").split("/")[-1]

        if repo_name.endswith(".git"):
            repo_name = repo_name[:-4]

        return repo_name or "repository"

    def clone_repository(self, artifact_id, uri):
        repo_name = self.repo_name_from_uri(uri)
        target_dir = self.download_dir / repo_name

        if target_dir.exists():
            print(f"📁 Repository already exists: {target_dir}")
            return target_dir

        print(f"⬇️ Cloning repository: {uri}")

        try:
            result = subprocess.run(
                ["git", "clone", "--depth", "1", uri, str(target_dir)],
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
                timeout=120
            )

            if result.returncode != 0:
                print(f"❌ Failed to clone repository for {artifact_id}")
                print(result.stderr.strip())
                return None

            print(f"✅ Cloned to: {target_dir}")
            return target_dir

        except Exception as e:
            print(f"❌ Clone error for {artifact_id}: {e}")
            return None

    def read_text_file(self, path, max_chars=200000):
        try:
            with open(path, "r", encoding="utf-8", errors="ignore") as file:
                return file.read(max_chars)
        except Exception:
            return ""

    def find_files_by_extensions(self, repo_dir, extensions):
        matches = []

        for root, dirs, files in os.walk(repo_dir):
            dirs[:] = [
                d for d in dirs
                if d not in {
                    ".git",
                    "__pycache__",
                    ".pytest_cache",
                    ".mypy_cache",
                    "node_modules",
                    ".venv",
                    "venv"
                }
            ]

            for file in files:
                if any(file.lower().endswith(ext) for ext in extensions):
                    matches.append(Path(root) / file)

        return matches

    def find_files_by_names(self, repo_dir, names):
        matches = []
        names_lower = {name.lower() for name in names}

        for root, dirs, files in os.walk(repo_dir):
            dirs[:] = [
                d for d in dirs
                if d not in {
                    ".git",
                    "__pycache__",
                    ".pytest_cache",
                    ".mypy_cache",
                    "node_modules",
                    ".venv",
                    "venv"
                }
            ]

            for file in files:
                if file.lower() in names_lower:
                    matches.append(Path(root) / file)

        return matches

    def collect_relevant_text(self, repo_dir):
        relevant_extensions = [".py", ".ipynb", ".md", ".rst", ".txt", ".yml", ".yaml", ".json", ".toml"]
        files = self.find_files_by_extensions(repo_dir, relevant_extensions)

        combined_text = ""
        selected_files = []

        for path in files[:300]:
            try:
                size_mb = path.stat().st_size / (1024 * 1024)
            except OSError:
                continue

            if size_mb > 5:
                continue

            selected_files.append(path)

            try:
                relative = path.relative_to(repo_dir)
            except Exception:
                relative = path

            combined_text += f"\n\n--- FILE: {relative} ---\n\n"
            combined_text += self.read_text_file(path, max_chars=50000)

        return combined_text, selected_files

    def count_python_scripts(self, repo_dir):
        scripts = self.find_files_by_extensions(repo_dir, [".py"])
        return len(scripts), scripts

    def count_notebooks(self, repo_dir):
        notebooks = self.find_files_by_extensions(repo_dir, [".ipynb"])
        return len(notebooks), notebooks

    def detect_workflow_files(self, repo_dir):
        workflow_names = {
            "makefile",
            "snakefile",
            "dvc.yaml",
            "dvc.lock",
            "tox.ini",
            "noxfile.py",
            "pyproject.toml",
            "setup.py",
            "environment.yml",
            "requirements.txt",
            "dockerfile",
            "docker-compose.yml",
            "docker-compose.yaml"
        }

        found = self.find_files_by_names(repo_dir, workflow_names)

        github_workflows = repo_dir / ".github" / "workflows"
        if github_workflows.exists():
            found.extend(self.find_files_by_extensions(github_workflows, [".yml", ".yaml"]))

        return found

    def detect_config_files(self, repo_dir):
        config_extensions = [".json", ".yaml", ".yml", ".toml", ".ini", ".cfg"]
        config_files = self.find_files_by_extensions(repo_dir, config_extensions)

        filtered = []

        for path in config_files:
            name = path.name.lower()

            if name in {
                "package-lock.json",
                "poetry.lock",
                "pdm.lock",
                "uv.lock"
            }:
                continue

            filtered.append(path)

        return filtered

    def detect_output_or_cache_dirs(self, repo_dir):
        candidate_names = {
            "cache",
            ".cache",
            "cached",
            "checkpoints",
            "checkpoint",
            "outputs",
            "output",
            "results",
            "result",
            "intermediate",
            "tmp",
            "temp",
            "logs",
            "runs",
            "artifacts",
            "models"
        }

        found = []

        for root, dirs, files in os.walk(repo_dir):
            dirs[:] = [
                d for d in dirs
                if d not in {
                    ".git",
                    "__pycache__",
                    ".pytest_cache",
                    ".mypy_cache",
                    "node_modules",
                    ".venv",
                    "venv"
                }
            ]

            for directory in dirs:
                if directory.lower() in candidate_names:
                    path = Path(root) / directory
                    try:
                        found.append(str(path.relative_to(repo_dir)))
                    except Exception:
                        found.append(str(path))

        return found

    def detect_keywords(self, text):
        text_lower = text.lower()

        keyword_groups = {
            "caching": [
                "cache",
                "cached",
                "caching",
                "joblib.memory",
                "lru_cache",
                "functools.cache",
                "diskcache",
                "memoize",
                "memoization"
            ],
            "checkpointing": [
                "checkpoint",
                "checkpointing",
                "save_checkpoint",
                "load_checkpoint",
                "resume_from_checkpoint",
                "resume",
                "restart",
                "restore"
            ],
            "parameterization": [
                "config",
                "configuration",
                "parameter",
                "parameters",
                "argparse",
                "click.option",
                "yaml",
                "toml",
                "json config",
                "--config",
                "--input",
                "--output"
            ],
            "workflow_modularity": [
                "makefile",
                "snakefile",
                "snakemake",
                "nextflow",
                "dvc",
                "pipeline",
                "workflow",
                "stage",
                "task",
                "step"
            ],
            "intermediate_outputs": [
                "intermediate",
                "output_dir",
                "results_dir",
                "save",
                "savefig",
                "to_csv",
                "to_json",
                "pickle.dump",
                "np.save",
                "torch.save",
                "model.save"
            ],
            "incremental_execution": [
                "incremental",
                "skip existing",
                "skip_existing",
                "if exists",
                "exists()",
                "is_file()",
                "isdir",
                "overwrite",
                "force",
                "rerun"
            ]
        }

        findings = {}

        for group, keywords in keyword_groups.items():
            found = []

            for keyword in keywords:
                if keyword in text_lower:
                    found.append(keyword)

            findings[group] = found

        return findings

    def evaluate_recomputation_avoidance(self, repo_dir, artifact_data):
        config = artifact_data.get("recomputation_avoidance", {})
        minimum_score = config.get("minimum_score", self.minimum_score)

        text, selected_files = self.collect_relevant_text(repo_dir)
        keyword_findings = self.detect_keywords(text)

        python_count, python_scripts = self.count_python_scripts(repo_dir)
        notebook_count, notebooks = self.count_notebooks(repo_dir)
        workflow_files = self.detect_workflow_files(repo_dir)
        config_files = self.detect_config_files(repo_dir)
        output_cache_dirs = self.detect_output_or_cache_dirs(repo_dir)

        score = 0
        evidence = []
        missing = []

        if keyword_findings["caching"]:
            score += 1
            evidence.append(f"Caching evidence found: {', '.join(keyword_findings['caching'][:8])}")
        else:
            missing.append("No caching evidence detected")

        if keyword_findings["checkpointing"]:
            score += 1
            evidence.append(f"Checkpoint/resume evidence found: {', '.join(keyword_findings['checkpointing'][:8])}")
        else:
            missing.append("No checkpoint/resume evidence detected")

        if keyword_findings["parameterization"] or len(config_files) > 0:
            score += 1
            if keyword_findings["parameterization"]:
                evidence.append(f"Parameterization evidence found: {', '.join(keyword_findings['parameterization'][:8])}")
            if config_files:
                short_configs = [str(p.relative_to(repo_dir)) for p in config_files[:8]]
                evidence.append(f"Configuration files found: {', '.join(short_configs)}")
        else:
            missing.append("No parameterization/configuration evidence detected")

        if keyword_findings["workflow_modularity"] or len(workflow_files) > 0:
            score += 1
            if keyword_findings["workflow_modularity"]:
                evidence.append(f"Workflow/modularity terms found: {', '.join(keyword_findings['workflow_modularity'][:8])}")
            if workflow_files:
                short_workflows = [str(p.relative_to(repo_dir)) for p in workflow_files[:8]]
                evidence.append(f"Workflow/build files found: {', '.join(short_workflows)}")
        else:
            missing.append("No workflow/modular execution evidence detected")

        if keyword_findings["intermediate_outputs"] or output_cache_dirs:
            score += 1
            if keyword_findings["intermediate_outputs"]:
                evidence.append(f"Intermediate output evidence found: {', '.join(keyword_findings['intermediate_outputs'][:8])}")
            if output_cache_dirs:
                evidence.append(f"Output/cache directories found: {', '.join(output_cache_dirs[:8])}")
        else:
            missing.append("No intermediate output evidence detected")

        if keyword_findings["incremental_execution"]:
            score += 1
            evidence.append(f"Incremental/rerun evidence found: {', '.join(keyword_findings['incremental_execution'][:8])}")
        else:
            missing.append("No incremental/rerun evidence detected")

        if python_count >= 3:
            score += 1
            evidence.append(f"Modular script structure detected: {python_count} Python files")
        else:
            missing.append("Weak modular script structure")

        if notebook_count > 0:
            score += 1
            evidence.append(f"Notebook-based executable workflow detected: {notebook_count} notebooks")

        recomputation_avoidant = score >= minimum_score

        return {
            "recomputation_avoidant": recomputation_avoidant,
            "score": score,
            "minimum_score": minimum_score,
            "python_file_count": python_count,
            "notebook_count": notebook_count,
            "workflow_file_count": len(workflow_files),
            "config_file_count": len(config_files),
            "output_cache_dirs": output_cache_dirs[:20],
            "keyword_findings": keyword_findings,
            "evidence": evidence,
            "missing": missing
        }

    def check_artifact(self, artifact_id, artifact_data):
        title = artifact_data.get("title", "")
        uri = artifact_data.get("uri", "")

        print("\n" + "=" * 80)
        print(f"🔍 Recomputation Avoidance Check for {artifact_id}")
        print(f"📦 Title: {title}")
        print(f"🔗 URI: {uri}")

        artifact_result = {
            "artifact_id": artifact_id,
            "title": title,
            "uri": uri,
            "recomputation_avoidant": False,
            "status": "failed"
        }

        if not self.is_git_repository(uri):
            print("❌ Unsupported artifact type for this checker.")
            artifact_result["reason"] = "Unsupported artifact type."
            return artifact_result

        repo_dir = self.clone_repository(artifact_id, uri)

        if repo_dir is None:
            artifact_result["reason"] = "Repository could not be cloned."
            artifact_result["status"] = "not_evaluated_repository_unavailable"
            return artifact_result

        result = self.evaluate_recomputation_avoidance(repo_dir, artifact_data)
        artifact_result.update(result)

        print("\n📊 Recomputation avoidance evidence:")
        print(f" - Score: {result['score']} / required {result['minimum_score']}")
        print(f" - Python files: {result['python_file_count']}")
        print(f" - Notebooks: {result['notebook_count']}")
        print(f" - Workflow files: {result['workflow_file_count']}")
        print(f" - Config files: {result['config_file_count']}")
        print(f" - Output/cache dirs: {', '.join(result['output_cache_dirs']) if result['output_cache_dirs'] else 'None'}")

        print("\n🔎 Evidence found:")
        if result["evidence"]:
            for item in result["evidence"]:
                print(f" - {item}")
        else:
            print(" - No recomputation-avoidance evidence found.")

        print("\n⚠️ Missing or weak evidence:")
        if result["missing"]:
            for item in result["missing"]:
                print(f" - {item}")
        else:
            print(" - No major missing evidence detected.")

        if result["recomputation_avoidant"]:
            artifact_result["status"] = "passed"
            print("\n✅ Recomputation Avoidance Result: PASSED")
        else:
            artifact_result["status"] = "failed"
            print("\n❌ Recomputation Avoidance Result: FAILED")

        return artifact_result

    def run(self):
        self.results = []

        print("🌱 Starting Recomputation Avoidance Fitness Function")
        print(f"📄 Metadata file: {self.json_file}")
        print(f"📁 Download directory: {self.download_dir}")

        for artifact_id, artifact_data in self.artifacts.items():
            result = self.check_artifact(artifact_id, artifact_data)
            self.results.append(result)

        print("\n" + "=" * 80)
        print("📌 Recomputation Avoidance Summary")
        print("=" * 80)

        for result in self.results:
            icon = "✅" if result["recomputation_avoidant"] else "❌"
            print(f"{icon} {result['artifact_id']}: {result['status']}")

        return self.results

In [3]:
checker = RecomputationAvoidanceChecker(
    json_file="artifacts.json",
    download_dir="downloads",
    minimum_score=4
)

recomputation_results = checker.run()

🌱 Starting Recomputation Avoidance Fitness Function
📄 Metadata file: artifacts.json
📁 Download directory: downloads

🔍 Recomputation Avoidance Check for artifact_1
📦 Title: We provide our resources in a dedicated repository
🔗 URI: https://github.com/hihey54/hicss58
📁 Repository already exists: downloads/hicss58

📊 Recomputation avoidance evidence:
 - Score: 3 / required 4
 - Python files: 5
 - Notebooks: 0
 - Workflow files: 1
 - Config files: 0
 - Output/cache dirs: None

🔎 Evidence found:
 - Parameterization evidence found: parameter, parameters
 - Workflow/build files found: requirements.txt
 - Modular script structure detected: 5 Python files

⚠️ Missing or weak evidence:
 - No caching evidence detected
 - No checkpoint/resume evidence detected
 - No intermediate output evidence detected
 - No incremental/rerun evidence detected

❌ Recomputation Avoidance Result: FAILED

🔍 Recomputation Avoidance Check for artifact_2
📦 Title: Trending Customer Dataset
🔗 URI: https://github.com/gith

In [4]:
if PANDAS_AVAILABLE:
    df = pd.DataFrame(recomputation_results)

    columns_to_show = [
        "artifact_id",
        "title",
        "recomputation_avoidant",
        "status",
        "score",
        "minimum_score",
        "python_file_count",
        "notebook_count",
        "workflow_file_count",
        "config_file_count"
    ]

    existing_columns = [col for col in columns_to_show if col in df.columns]
    display(df[existing_columns])
else:
    for result in recomputation_results:
        print(result)

,artifact_id,title,recomputation_avoidant,status,score,minimum_score,python_file_count,notebook_count,workflow_file_count,config_file_count
0,artifact_1,We provide our resources in a dedicated reposi...,False,failed,3.0,4.0,5.0,0.0,1.0,0.0
1,artifact_2,Trending Customer Dataset,False,not_evaluated_repository_unavailable,NaN,NaN,NaN,NaN,NaN,NaN
2,artifact_3,Python algorithms,True,passed,7.0,4.0,1381.0,0.0,8.0,20.0
3,artifact_4,Scikit-learn,True,passed,7.0,4.0,997.0,0.0,29.0,53.0
4,artifact_5,Pandas,True,passed,8.0,4.0,1509.0,1.0,13.0,55.0
5,artifact_6,NumPy,True,passed,7.0,4.0,494.0,0.0,30.0,50.0
6,artifact_7,Matplotlib,True,passed,8.0,4.0,913.0,3.0,24.0,54.0
7,artifact_8,Scrapy,True,passed,7.0,4.0,437.0,0.0,10.0,13.0
8,artifact_9,Flask,True,passed,7.0,4.0,83.0,0.0,11.0,15.0
9,artifact_10,TensorFlow,True,passed,8.0,4.0,3158.0,34.0,28.0,109.0


In [5]:
output_file = "recomputation_avoidance_results.json"

with open(output_file, "w", encoding="utf-8") as file:
    json.dump(recomputation_results, file, indent=4)

print(f"✅ Results saved to {output_file}")

✅ Results saved to recomputation_avoidance_results.json
